# Explainable Multimodal model (fMRI+MRI) for Alzheimer's disease diagnosis

In [ ]:
# --------------------- CONFIG ---------------------
LABELS_PATH     = r"H:\Master's Thesis\Full dataset\derivatives\full dataset.csv"
SMRI_FEATS_PATH = r"H:\Master's Thesis\Full dataset\derivatives\fs_features.csv"
FMRI_FEATS_PATH = r"H:\Master's Thesis\Full dataset\derivatives\fmri_features.csv"
OUTPUT_DIR      = r"H:\Master's Thesis\Results\Multimodal_Hybrid_OvO_PlusPlus"

# label/covariate column candidates in labels file
LABELS_SUBJECT_COL_CANDIDATES = ["Subject ID", "subject_id", "ID", "SubjectID"]
LABELS_LABEL_COL_CANDIDATES   = ["label", "dx", "diagnosis", "Diagnosis", "DX"]
LABELS_AGE_COL_CANDIDATES     = ["Scan_Age", "Age", "age"]
LABELS_SEX_COL_CANDIDATES     = ["Sex", "sex", "Gender", "gender"]
LABELS_BATCH_COL_CANDIDATES   = ["batch", "Phase", "site", "Site"]

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 3
MASTER_SEED   = 42

# Modal screening (RAM-safe)
SMRI_KBEST    = 600
FMRI_KBEST    = 900
SMRI_PCA_GRID = [20, 30]
FMRI_PCA_GRID = [30, 50]

# Base learners
SVC_C_GRID    = [0.05, 0.1, 0.5, 1.0, 3.0]
CALIB_METHOD  = "sigmoid"

# Nyström (only for AD↔MCI head)
NYSTROEM_COMPONENTS = [200, 400]
NYSTROEM_GAMMA      = [1e-3, 3e-3, 1e-2]

# Preprocessing
MISSING_MAX_FRAC = 0.6
VAR_THRESHOLD    = 1e-8
WINSOR_Q         = 0.005

# --------------------- IMPORTS ---------------------
import os, json, time, datetime, re, warnings, gc
import numpy as np
import pandas as pd
from typing import List, Optional, Tuple
from contextlib import contextmanager

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    precision_recall_curve, auc, brier_score_loss
)
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220)
np.set_printoptions(suppress=True)

# --------------------- LOGGING ---------------------
def now(): return time.strftime("%H:%M:%S")
def log(msg): print(f"[{now()}] {msg}", flush=True)

@contextmanager
def timed(section):
    t0 = time.time()
    log(f"▶ {section} ...")
    try:
        yield
    finally:
        log(f"✓ {section} done in {time.time()-t0:.1f}s")

def ensure_dir(path: str): os.makedirs(path, exist_ok=True); return path
def timestamp(): return datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# --------------------- HELPERS ---------------------
def pick_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns: return c
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower: return lower[c.lower()]
    return None

def pr_auc_macro(y_true, probas, class_order):
    aucs = []
    present = [c for c in class_order if np.any(y_true == c)]
    for i, c in enumerate(class_order):
        if c not in present: continue
        yb = (y_true == c).astype(int)
        p, r, _ = precision_recall_curve(yb, probas[:, i])
        aucs.append(auc(r, p))
    return float(np.mean(aucs)) if aucs else np.nan

def aggregate_pairwise_probs(classes, pairwise_dict, n_samples):
    """
    Simple pairwise-probability coupling:
      For each pair (i,j) we have probabilities over {i,j}: p_i, p_j.
      We build class scores by summing p for that class across its pairs, then average.
      Finally renormalize so rows sum to 1.
    pairwise_dict: {('AD','CN'): proba_2col_in_class_order, ...}
      each is shape (n_samples, 2) aligned to the test set of the fold.
    Returns probas (n_samples, n_classes) in the order of `classes`.
    """
    idx = {c:i for i,c in enumerate(classes)}
    scores = np.zeros((n_samples, len(classes)), dtype=float)
    counts = np.zeros(len(classes), dtype=float)
    for (a,b), P in pairwise_dict.items():
        ia, ib = idx[a], idx[b]
        scores[:, ia] += P[:, 0]
        scores[:, ib] += P[:, 1]
        counts[ia] += 1; counts[ib] += 1
    counts[counts==0] = 1.0
    scores = scores / counts
    s = scores.sum(axis=1, keepdims=True); s[s==0]=1.0
    return scores / s

# --------------------- TRANSFORMERS ---------------------
class ColumnSelect(BaseEstimator, TransformerMixin):
    def __init__(self, columns: List[str]): self.columns = columns
    def fit(self, X, y=None):
        if not isinstance(X, pd.DataFrame): raise ValueError("ColumnSelect expects DataFrame")
        miss = [c for c in self.columns if c not in X.columns]
        if miss: raise ValueError(f"ColumnSelect missing: {miss[:5]} ...")
        return self
    def transform(self, X): return X[self.columns]

class ToNumpy(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X): return X.values if isinstance(X, pd.DataFrame) else np.asarray(X)

class ToFloat32(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.values if isinstance(X, pd.DataFrame) else X
        return np.asarray(X, dtype=np.float32)

class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, q=0.005): self.q=q
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lower_ = np.nanquantile(X, self.q, axis=0)
        self.upper_ = np.nanquantile(X, 1-self.q, axis=0)
        return self
    def transform(self, X): return np.clip(np.asarray(X, float), self.lower_, self.upper_)

def try_import_neurocombat():
    try:
        from neurocombat_sklearn import CombatModel
        return CombatModel
    except Exception:
        return None

class OptionalComBat(BaseEstimator, TransformerMixin):
    def __init__(self, batch_series: Optional[pd.Series]=None):
        self.batch_series = batch_series
    def fit(self, X, y=None):
        self.model_ = None
        CombatModel = try_import_neurocombat()
        if CombatModel is None or self.batch_series is None: return self
        idx = X.index if isinstance(X, pd.DataFrame) else None
        b = self.batch_series.loc[idx].values if idx is not None else self.batch_series.values
        self.model_ = CombatModel()
        Xv = np.asarray(X if not isinstance(X, pd.DataFrame) else X.values, float)
        self.model_.fit(Xv, b)
        return self
    def transform(self, X):
        if self.model_ is None: return X
        idx = X.index if isinstance(X, pd.DataFrame) else None
        b = self.batch_series.loc[idx].values if idx is not None else self.batch_series.values
        Xv = np.asarray(X if not isinstance(X, pd.DataFrame) else X.values, float)
        Xt = self.model_.transform(Xv, b)
        return pd.DataFrame(Xt, index=(X.index if isinstance(X, pd.DataFrame) else None),
                            columns=(X.columns if isinstance(X, pd.DataFrame) else None))

class CovariateResidualizer(BaseEstimator, TransformerMixin):
    def __init__(self, cov_df: Optional[pd.DataFrame]=None, cov_cols: Optional[List[str]]=None):
        self.cov_df = cov_df; self.cov_cols = cov_cols
    def fit(self, X, y=None):
        cv = list(self.cov_cols) if self.cov_cols else []
        if not cv: self.ct_=None; self.model_=None; self.fimp_=None; return self
        if not isinstance(X, pd.DataFrame): raise ValueError("CovariateResidualizer expects DataFrame")
        Z = self.cov_df.loc[X.index, cv].copy()
        num = [c for c in cv if pd.api.types.is_numeric_dtype(Z[c])]
        cat = [c for c in cv if c not in num]
        num_tfm = Pipeline([("imp", SimpleImputer(strategy="median"))]) if num else "drop"
        cat_tfm = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]) if cat else "drop"
        self.ct_ = ColumnTransformer([("num", num_tfm, num), ("cat", cat_tfm, cat)], remainder="drop", sparse_threshold=0.0)
        Z_enc = self.ct_.fit_transform(Z)

        self.fimp_ = SimpleImputer(strategy="median", keep_empty_features=True)
        Xv = self.fimp_.fit_transform(X.values.astype(float, copy=False))

        self.model_ = Ridge(alpha=1e-3, fit_intercept=True)
        self.model_.fit(Z_enc, Xv)
        self.cols_ = list(X.columns)
        return self
    def transform(self, X):
        if getattr(self, "model_", None) is None: return X
        Z = self.cov_df.loc[X.index, self.cov_cols]
        Z_enc = self.ct_.transform(Z)
        Xv = self.fimp_.transform(X.values.astype(float, copy=False))
        pred = self.model_.predict(Z_enc)
        resid = Xv - pred
        if resid.shape[1] != X.shape[1]:
            if resid.shape[1] < X.shape[1]:
                pad = np.full((resid.shape[0], X.shape[1]-resid.shape[1]), np.nan, dtype=float)
                resid = np.concatenate([resid, pad], axis=1)
            else:
                resid = resid[:, :X.shape[1]]
        return pd.DataFrame(resid, index=X.index, columns=list(X.columns))

class DropHiMissing(BaseEstimator, TransformerMixin):
    def __init__(self, max_frac=0.6): self.max_frac = max_frac
    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            miss = X.isna().mean(axis=0).values
            cols = np.array(list(X.columns))
        else:
            X = np.asarray(X)
            miss = np.mean(~np.isfinite(X), axis=0)
            cols = np.arange(X.shape[1])
        self.keep_mask_ = miss <= float(self.max_frac)
        self.cols_ = cols[self.keep_mask_]
        return self
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.loc[:, self.cols_]
        return np.asarray(X)[:, self.keep_mask_]

# --------------------- LOAD DATA ---------------------
def load_features_with_prefix(path, id_candidates: List[str], prefix: str):
    df = pd.read_csv(path)
    id_col = pick_col(df, id_candidates)
    if id_col is None:
        raise ValueError(f"Could not find a subject ID column in {path}. Tried: {id_candidates}")
    df = df.rename(columns={id_col: "subject_id"})
    nonfeat = {"label","dx","diagnosis","age","sex","gender","phase","site","batch","mean_fd","fd","eTIV","etiv","intracranialvol","icv"}
    feat_cols = [c for c in df.columns if c != "subject_id" and c.lower() not in nonfeat]
    num_cols  = [c for c in feat_cols if pd.api.types.is_numeric_dtype(df[c])]
    dropped   = [c for c in feat_cols if c not in num_cols]
    if dropped[:3]: log(f"⚠ {prefix}: dropped {len(dropped)} non-numeric cols (e.g., {dropped[:3]})")
    feat = df[["subject_id"] + num_cols].copy()
    for c in num_cols:
        feat[c] = pd.to_numeric(feat[c], errors="coerce").astype(np.float32)
    feat = feat.rename(columns={c: f"{c}_{prefix}" for c in num_cols})
    return feat

with timed("Loading labels/covariates"):
    labels_raw = pd.read_csv(LABELS_PATH)
    sub_col   = pick_col(labels_raw, LABELS_SUBJECT_COL_CANDIDATES)
    lab_col   = pick_col(labels_raw, LABELS_LABEL_COL_CANDIDATES)
    age_col   = pick_col(labels_raw, LABELS_AGE_COL_CANDIDATES)
    sex_col   = pick_col(labels_raw, LABELS_SEX_COL_CANDIDATES)
    batch_col = pick_col(labels_raw, LABELS_BATCH_COL_CANDIDATES)
    assert sub_col is not None, "No subject ID column in labels file."
    assert lab_col is not None, "No label/diagnosis column in labels file."
    labels = labels_raw[[c for c in [sub_col, lab_col, age_col, sex_col, batch_col] if c is not None]].copy()
    labels.columns = ["subject_id"] + [c for c in labels.columns if c != sub_col]
    labels = labels.rename(columns={lab_col: "label"})
    if age_col and age_col in labels.columns: labels = labels.rename(columns={age_col: "age"})
    if sex_col and sex_col in labels.columns: labels = labels.rename(columns={sex_col: "sex"})
    if batch_col and batch_col in labels.columns: labels = labels.rename(columns={batch_col: "batch"})
    labels["label"] = labels["label"].astype(str).str.upper().str.replace(r"\s+", "", regex=True)

FEATURE_ID_CANDS = ["subject_id","Subject ID","SUBJECT","ID","participant_id","PatientID"]

with timed("Loading sMRI features"):
    smri = load_features_with_prefix(SMRI_FEATS_PATH, FEATURE_ID_CANDS, "smri")

with timed("Loading fMRI features"):
    fmri = load_features_with_prefix(FMRI_FEATS_PATH, FEATURE_ID_CANDS, "fmri")

with timed("Augmenting covariates from features (eTIV, mean_FD)"):
    etiv_candidates = [c for c in smri.columns if re.search(r"(etiv|intracranial|EstimatedTotalIntraCranialVol)", c, flags=re.I)]
    labels = labels.merge(smri[["subject_id", etiv_candidates[0]]].rename(columns={etiv_candidates[0]:"eTIV"}), on="subject_id", how="left") if etiv_candidates else labels.assign(eTIV=np.nan)
    fd_candidates = [c for c in fmri.columns if re.fullmatch(r"(mean_)?FD(_fmri)?", c, flags=re.I)]
    labels = labels.merge(fmri[["subject_id", fd_candidates[0]]].rename(columns={fd_candidates[0]:"mean_FD"}), on="subject_id", how="left") if fd_candidates else labels.assign(mean_FD=np.nan)

with timed("Merging subjects with BOTH modalities"):
    df = labels.merge(smri, on="subject_id", how="inner").merge(fmri, on="subject_id", how="inner")
    log(f"Subjects with BOTH modalities: {len(df)} (from labels={len(labels)}, smri={len(smri)}, fmri={len(fmri)})")

exclude_cols = {"subject_id","label","age","sex","eTIV","mean_FD","batch"}
smri_feats = [c for c in df.columns if c.endswith("_smri") and c not in exclude_cols]
fmri_feats = [c for c in df.columns if c.endswith("_fmri") and c not in exclude_cols]
assert len(smri_feats) > 0 and len(fmri_feats) > 0

cov_cols_present = [c for c in ["age","sex","eTIV","mean_FD","batch"] if c in df.columns]
cov_df   = df[cov_cols_present].copy()
smri_cov = cov_df.copy()
fmri_cov = cov_df.copy()
batch_series = df["batch"] if "batch" in df.columns else None

log(f"Covariates used: {cov_cols_present}")
log(f"sMRI features: {len(smri_feats)} | fMRI features: {len(fmri_feats)}")

# --------------------- EMBEDDERS ---------------------
def build_smri_embed():
    return Pipeline([
        ("select", ColumnSelect(smri_feats)),
        ("resid",  CovariateResidualizer(cov_df=smri_cov, cov_cols=[c for c in ["age","sex","eTIV"] if c in cov_cols_present])),
        ("combat", OptionalComBat(batch_series)),
        ("dropmiss", DropHiMissing(max_frac=MISSING_MAX_FRAC)),
        ("to_np",  ToNumpy()),
        ("imp",    SimpleImputer(strategy="median", keep_empty_features=True)),
        ("to32a",  ToFloat32()),
        ("winsor", Winsorizer(q=WINSOR_Q)),
        ("varth",  VarianceThreshold(threshold=VAR_THRESHOLD)),
        ("scale",  StandardScaler(with_mean=True, with_std=True)),
        ("to32b",  ToFloat32()),
        ("kbest",  SelectKBest(score_func=f_classif, k=min(SMRI_KBEST, max(20, len(smri_feats))))),
        ("pca",    PCA(n_components=30, svd_solver="randomized", random_state=MASTER_SEED)),
        ("to32c",  ToFloat32()),
    ])

def build_fmri_embed():
    return Pipeline([
        ("select", ColumnSelect(fmri_feats)),
        ("resid",  CovariateResidualizer(cov_df=fmri_cov, cov_cols=[c for c in ["age","sex","mean_FD"] if c in cov_cols_present])),
        ("combat", OptionalComBat(batch_series)),
        ("dropmiss", DropHiMissing(max_frac=MISSING_MAX_FRAC)),
        ("to_np",  ToNumpy()),
        ("imp",    SimpleImputer(strategy="median", keep_empty_features=True)),
        ("to32a",  ToFloat32()),
        ("winsor", Winsorizer(q=WINSOR_Q)),
        ("varth",  VarianceThreshold(threshold=VAR_THRESHOLD)),
        ("scale",  StandardScaler(with_mean=True, with_std=True)),
        ("to32b",  ToFloat32()),
        ("kbest",  SelectKBest(score_func=f_classif, k=min(FMRI_KBEST, max(20, len(fmri_feats))))),
        ("pca",    PCA(n_components=50, svd_solver="randomized", random_state=MASTER_SEED)),
        ("to32c",  ToFloat32()),
    ])

def build_fused_embed():
    return FeatureUnion([("smri", build_smri_embed()), ("fmri", build_fmri_embed())])

# --------------------- PAIRWISE HEADS ---------------------
def make_linear_head():
    base = LinearSVC(C=1.0, penalty="l2", loss="squared_hinge", dual=True,
                     class_weight="balanced", max_iter=10000, random_state=MASTER_SEED)
    clf  = CalibratedClassifierCV(estimator=base, cv=3, method=CALIB_METHOD)
    return clf

def make_nystroem_head(n_components=300, gamma=1e-3):
    # Nyström(RBF) → LinearSVC → Calibrated
    feature_map = Nystroem(kernel="rbf", n_components=n_components, gamma=gamma, random_state=MASTER_SEED)
    base = LinearSVC(C=1.0, penalty="l2", loss="squared_hinge", dual=True,
                     class_weight="balanced", max_iter=10000, random_state=MASTER_SEED)
    head = Pipeline([("nyst", feature_map), ("svc", base)])
    clf  = CalibratedClassifierCV(estimator=head, cv=3, method=CALIB_METHOD)
    return clf

def build_pair_pipeline(pair):
    """
    pair: tuple like ('AD','CN'), ('AD','MCI'), ('CN','MCI')
    Returns estimator Pipeline with fused embed + pairwise head.
    By design: AD↔MCI uses Nyström head, others use linear.
    """
    embed = build_fused_embed()
    if set(pair) == {"AD","MCI"}:
        clf = make_nystroem_head()  # params tuned via grid
        pipe = Pipeline([("embed", embed), ("clf", clf)])
    else:
        clf = make_linear_head()
        pipe = Pipeline([("embed", embed), ("clf", clf)])
    return pipe

def grid_for_pair(pair):
    grid = {
        "embed__smri__pca__n_components": SMRI_PCA_GRID,
        "embed__fmri__pca__n_components": FMRI_PCA_GRID,
    }
    if set(pair) == {"AD","MCI"}:
        # Tune Nyström + SVC C
        grid.update({
            "clf__estimator__nyst__n_components": NYSTROEM_COMPONENTS,
            "clf__estimator__nyst__gamma": NYSTROEM_GAMMA,
            "clf__estimator__svc__C": SVC_C_GRID,
        })
    else:
        grid.update({
            "clf__estimator__C": SVC_C_GRID,
        })
    return grid

# --------------------- TRAIN / EVAL ---------------------
def fit_pairwise(X, y, pair):
    a, b = pair
    m = (y==a)|(y==b)
    Xp, yp = X[m], y[m]
    pipe = build_pair_pipeline(pair)
    g = grid_for_pair(pair)
    inner = StratifiedKFold(n_splits=N_INNER_FOLDS, shuffle=True, random_state=MASTER_SEED)
    search = GridSearchCV(
        estimator=pipe, param_grid=g, scoring="roc_auc", n_jobs=1, cv=inner, refit=True, verbose=0
    )
    search.fit(Xp, yp)
    return search.best_estimator_, search.best_params_

def run_outer_cv(df, outdir):
    ensure_dir(outdir)
    y_all = df["label"].values
    subj_all = df["subject_id"].values
    classes = ["AD","CN","MCI"]
    assert set(np.unique(y_all)) == set(classes), f"Expected labels {classes}"

    outer = StratifiedKFold(n_splits=N_OUTER_FOLDS, shuffle=True, random_state=MASTER_SEED)
    tsdir = ensure_dir(os.path.join(outdir, timestamp()))
    ensure_dir(os.path.join(tsdir, "splits")); ensure_dir(os.path.join(tsdir, "metrics"))

    spec = {
        "n": len(df), "classes": classes,
        "smri_n_features": len(smri_feats), "fmri_n_features": len(fmri_feats),
        "covariates_present": [c for c in ["age","sex","eTIV","mean_FD","batch"] if c in df.columns],
        "grids": {
            "SMRI_PCA": SMRI_PCA_GRID, "FMRI_PCA": FMRI_PCA_GRID,
            "C": SVC_C_GRID, "NYSTROEM_COMPONENTS": NYSTROEM_COMPONENTS, "NYSTROEM_GAMMA": NYSTROEM_GAMMA
        },
        "calibration": CALIB_METHOD
    }
    with open(os.path.join(tsdir, "spec.json"), "w", encoding="utf-8") as f: json.dump(spec, f, indent=2)

    pairs = [("AD","CN"), ("AD","MCI"), ("CN","MCI")]
    fold_rows = []

    with timed("Full nested CV run — Hybrid OvO++"):
        for k, (tr, te) in enumerate(outer.split(np.zeros(len(y_all)), y_all), start=1):
            with timed(f"===== Outer Fold {k}/{N_OUTER_FOLDS} ====="):
                X_tr, X_te = df.iloc[tr], df.iloc[te]
                y_tr, y_te = y_all[tr], y_all[te]
                log(f"Train class counts: {dict(pd.Series(y_tr).value_counts())} | Test class counts: {dict(pd.Series(y_te).value_counts())}")

                bests = {}
                pair_metrics = {}
                pair_probs = {}

                # ---- Fit three pairwise heads
                for p in pairs:
                    with timed(f"  Fit pair {p[0]} vs {p[1]}"):
                        best_est, best_params = fit_pairwise(X_tr, y_tr, p)
                        # store and log
                        bests[p] = best_est
                        log(f"    Best params {p[0]}vs{p[1]}: {best_params}")

                        # evaluate on test subset belonging to that pair
                        mte = (y_te==p[0]) | (y_te==p[1])
                        if np.any(mte):
                            y_true = y_te[mte]
                            P = best_est.predict_proba(X_te[mte])  # columns align with best_est.classes_
                            # reorder columns to [p[0], p[1]]
                            order = [list(best_est.classes_).index(p[0]), list(best_est.classes_).index(p[1])]
                            P_pair = P[:, order]
                            pair_probs[p] = np.zeros((len(y_te), 2), dtype=float)
                            pair_probs[p][mte] = P_pair
                            acc_pair = accuracy_score(y_true, np.where(P_pair[:,0]>=0.5, p[0], p[1]))
                            try:
                                auc_pair = roc_auc_score((y_true==p[0]).astype(int), P_pair[:,0])
                            except Exception:
                                auc_pair = np.nan
                            pair_metrics[p] = {"acc": float(acc_pair), "auc": float(auc_pair)}
                        else:
                            pair_probs[p] = np.zeros((len(y_te),2), dtype=float)
                            pair_metrics[p] = {"acc": np.nan, "auc": np.nan}

                # ---- Aggregate pairwise probabilities to multiclass
                Y_proba = aggregate_pairwise_probs(classes, pair_probs, len(y_te))
                y_pred = np.array([classes[i] for i in np.argmax(Y_proba, axis=1)])

                # ---- Metrics
                majority_label = pd.Series(y_tr).mode()[0]
                baseline_acc = float(np.mean(y_te == majority_label))
                acc  = accuracy_score(y_te, y_pred)
                f1m  = f1_score(y_te, y_pred, average="macro")
                try:
                    roc_ovo = roc_auc_score(y_te, Y_proba[:, [classes.index(c) for c in np.unique(y_te)]],
                                            multi_class="ovo", labels=np.unique(y_te))
                except Exception:
                    roc_ovo = np.nan
                prauc = pr_auc_macro(pd.Series(y_te), Y_proba, classes)
                brier = float(np.mean([
                    brier_score_loss((pd.Series(y_te)==cls).astype(int).values, Y_proba[:,classes.index(cls)])
                    for cls in classes if np.any(pd.Series(y_te)==cls)
                ]))
                cm = confusion_matrix(y_te, y_pred, labels=classes)

                # Record
                row = {
                    "fold": k, "n_train": int(len(tr)), "n_test": int(len(te)),
                    "baseline_acc": baseline_acc,
                    "accuracy": acc, "f1_macro": f1m, "roc_auc_ovo": roc_ovo, "pr_auc_macro": prauc, "brier_mean": brier,
                    "acc_AD_vs_CN": pair_metrics[("AD","CN")]["acc"],
                    "auc_AD_vs_CN": pair_metrics[("AD","CN")]["auc"],
                    "acc_AD_vs_MCI": pair_metrics[("AD","MCI")]["acc"],
                    "auc_AD_vs_MCI": pair_metrics[("AD","MCI")]["auc"],
                    "acc_CN_vs_MCI": pair_metrics[("CN","MCI")]["acc"],
                    "auc_CN_vs_MCI": pair_metrics[("CN","MCI")]["auc"],
                    **{f"cm_{classes[i]}{classes[j]}": int(cm[i,j]) for i in range(3) for j in range(3)},
                }
                fold_rows.append(row)

                # Save per-fold artifacts
                fold_dir = ensure_dir(os.path.join(tsdir, "metrics"))
                pd.DataFrame({
                    "subject_id": df.iloc[te]["subject_id"].values,
                    "true": y_te, "pred": y_pred,
                    "proba_AD": Y_proba[:,0], "proba_CN": Y_proba[:,1], "proba_MCI": Y_proba[:,2],
                }).to_csv(os.path.join(fold_dir, f"fold_{k}_predictions.csv"), index=False)

                with open(os.path.join(tsdir, "splits", f"fold_{k}_indices.json"), "w") as f:
                    json.dump({"train_idx": list(map(int,tr)), "test_idx": list(map(int,te))}, f)

                for p in pairs:
                    with open(os.path.join(fold_dir, f"fold_{k}_best_{p[0]}vs{p[1]}.json"), "w") as f:
                        # GridSearchCV already logged via 'best params' line; we store just class order for safety.
                        json.dump({"classes_": list(bests[p].classes_) if hasattr(bests[p], "classes_") else list(p)}, f)

                log(f"Pairwise acc — ADvsCN: {row['acc_AD_vs_CN']:.3f} | ADvsMCI: {row['acc_AD_vs_MCI']:.3f} | CNvsMCI: {row['acc_CN_vs_MCI']:.3f}")
                log(f"Fold metrics — Acc: {acc:.3f} | F1_macro: {f1m:.3f} | ROC AUC ovo: {roc_ovo:.3f} | Brier: {brier:.3f}")

                del bests, pair_probs, Y_proba, y_pred, cm
                gc.collect()

    # ---- Aggregate ----
    folds_df = pd.DataFrame(fold_rows)
    agg = folds_df[["baseline_acc","accuracy","f1_macro","roc_auc_ovo","pr_auc_macro","brier_mean"]].agg(["mean","std"]).T.reset_index().rename(columns={"index":"metric"})
    pw_agg = folds_df[["acc_AD_vs_CN","auc_AD_vs_CN","acc_AD_vs_MCI","auc_AD_vs_MCI","acc_CN_vs_MCI","auc_CN_vs_MCI"]].agg(["mean","std"]).T.reset_index().rename(columns={"index":"pair_metric"})

    ensure_dir(os.path.join(tsdir,"metrics"))
    folds_df.to_csv(os.path.join(tsdir,"metrics","fold_metrics.csv"), index=False)
    agg.to_csv(os.path.join(tsdir,"metrics","aggregate_metrics.csv"), index=False)
    pw_agg.to_csv(os.path.join(tsdir,"metrics","aggregate_pairwise_metrics.csv"), index=False)

    log("\n==== Aggregate (mean ± sd) ====")
    for _, row in agg.iterrows():
        log(f"{row['metric']:>12}: {row['mean']:.3f} ± {row['std']:.3f}")

    log("\n==== Pairwise metrics (mean ± sd) ====")
    for _, row in pw_agg.iterrows():
        log(f"{row['pair_metric']:>14}: {row['mean']:.3f} ± {row['std']:.3f}")

    log(f"Artifacts saved to: {tsdir}")

# --------------------- RUN ---------------------
try:
    with timed("Merging subjects with BOTH modalities"):
        pass  # already merged above; keep timing symmetry
    with timed("Full nested CV run — Hybrid OvO (Linear for easy pairs; Nyström-RBF for AD↔MCI)"):
        run_outer_cv(df, OUTPUT_DIR)
except Exception:
    log("✖ ERROR during run. Traceback (last 3 frames):")
    import traceback; traceback.print_exc(limit=3)


## Explainablility analysis using SHAP

In [ ]:
# ====== Multimodal MRI+fMRI — Hybrid OvO++ with XAI (index-safe) ======
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.kernel_approximation import Nystroem
from sklearn.metrics import roc_auc_score
from inspect import signature

warnings.filterwarnings("ignore")

# --------------------------- CONFIG ---------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# File paths (edit if needed)
FS_CSV   = r"H:\Master's Thesis\Full dataset\derivatives\fs_features.csv"
FMR_CSV  = r"H:\Master's Thesis\Full dataset\derivatives\fmri_features.csv"
LAB_CSV  = r"H:\Master's Thesis\Full dataset\derivatives\full dataset.csv"

# Output dir
OUT_DIR = r"H:\Master's Thesis\Results\Multimodal_Hybrid_OvO_PlusPlus_XAI"
os.makedirs(OUT_DIR, exist_ok=True)

# Column name fallbacks (auto-detect but allow overrides)
ID_FALLBACKS     = ["subject_id","Subject ID","SUBJECT_ID","participant_id","Participant ID"]
TARGET_FALLBACKS = ["label","diagnosis","Diagnosis","dx","DX"]

# Hyperparam grids (kept moderate)
PCA_S_GRID = [20, 30]      # sMRI PCs
PCA_F_GRID = [30, 50]      # fMRI PCs

C_GRID_LIN  = [0.05, 0.1, 0.5, 1.0]        # LinearSVC C
NYST_NC     = [200]                         # Nyström components
NYST_GAMMA  = [0.001, 0.003, 0.01]          # Nyström RBF gamma
C_GRID_RBF  = [0.05, 0.5, 1.0, 3.0]         # LinearSVC C after Nyström

CALIB_CV    = 3                              # internal CV for probability calibration
BGNSZ       = 120                            # background size for Kernel SHAP
NSHZ        = 250                            # nsamples for Kernel SHAP

PAIR_HEADS  = [("AD","CN"), ("AD","MCI"), ("CN","MCI")]  # OvO

# --------------------------- UTIL: CalibratedClassifierCV compatibility ---------------------------
def make_calibrated_linear_svc(C=1.0, method="sigmoid", cv=CALIB_CV, random_state=RANDOM_SEED):
    base = LinearSVC(C=C, class_weight="balanced", random_state=random_state)
    sig = signature(CalibratedClassifierCV.__init__)
    if "estimator" in sig.parameters:
        return CalibratedClassifierCV(estimator=base, cv=cv, method=method)
    else:
        return CalibratedClassifierCV(base_estimator=base, cv=cv, method=method)

# --------------------------- LOAD & ALIGN ---------------------------
def _detect(series, candidates):
    for c in candidates:
        if c in series:
            return c
    return None

def load_tables():
    print(f"[LOG] Loading sMRI: {FS_CSV}")
    fs = pd.read_csv(FS_CSV)
    print(f"[LOG] Loading fMRI: {FMR_CSV}")
    fm = pd.read_csv(FMR_CSV)
    print(f"[LOG] Loading labels/cov: {LAB_CSV}")
    lab = pd.read_csv(LAB_CSV)

    # clean column names
    fs.columns = [c.strip() for c in fs.columns]
    fm.columns = [c.strip() for c in fm.columns]
    lab.columns = [c.strip() for c in lab.columns]

    id_fs  = _detect(fs.columns, ID_FALLBACKS)
    id_fm  = _detect(fm.columns, ID_FALLBACKS)
    id_lab = _detect(lab.columns, ID_FALLBACKS)
    tgt    = _detect(lab.columns, TARGET_FALLBACKS)

    assert id_fs and id_fm and id_lab, "Could not find a subject_id column in one or more files."
    assert tgt, "Could not find a label/diagnosis column in the labels file."

    print(f"[INFO] Detected ID: fs='{id_fs}', fm='{id_fm}', labels='{id_lab}' | TARGET='{tgt}'")

    # pick covariates if present (your file has Sex)
    age_col  = _detect(lab.columns, ["age","Age","AGE"])
    sex_col  = _detect(lab.columns, ["sex","Sex","SEX"])
    batchcol = _detect(lab.columns, ["batch","site","scanner","Batch","Site"])
    print(f"[INFO] Covariates in labels → age='{age_col}', sex='{sex_col}', batch='{batchcol}'")

    keep = [id_lab, tgt] + [c for c in [age_col, sex_col, batchcol] if c]
    lab2 = lab[keep].copy()

    fs = fs.rename(columns={id_fs:"subject_id"})
    fm = fm.rename(columns={id_fm:"subject_id"})
    lab2 = lab2.rename(columns={id_lab:"subject_id", tgt:"label"})

    common = set(fs["subject_id"]).intersection(set(fm["subject_id"])).intersection(set(lab2["subject_id"]))
    fs = fs[fs["subject_id"].isin(common)].sort_values("subject_id").reset_index(drop=True)
    fm = fm[fm["subject_id"].isin(common)].sort_values("subject_id").reset_index(drop=True)
    lab2 = lab2[lab2["subject_id"].isin(common)].sort_values("subject_id").reset_index(drop=True)

    fs = fs.set_index("subject_id")
    fm = fm.set_index("subject_id")
    lab2 = lab2.set_index("subject_id")

    fs_feats = [c for c in fs.columns if c != "label"]
    fm_feats = [c for c in fm.columns if c != "label"]

    y = lab2["label"].astype(str).copy()
    cov = lab2.drop(columns=["label"]).copy()

    print(f"[INFO] Subjects aligned: {len(y)} | Class counts: {dict(y.value_counts())}")
    print(f"[INFO] sMRI features: {len(fs_feats)} | fMRI features: {len(fm_feats)}")
    return fs, fm, y, cov, fs_feats, fm_feats

fs, fm, y_all, cov, FS_COLS, FM_COLS = load_tables()

# --------------------------- BRANCHES & EMBEDDER ---------------------------
def build_smri_branch(n_comp):
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("pca", PCA(n_components=n_comp, random_state=RANDOM_SEED))
    ])

def build_fmri_branch(n_comp):
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler()),
        ("pca", PCA(n_components=n_comp, random_state=RANDOM_SEED))
    ])

def make_embedder(n_smri, n_fmri):
    return ColumnTransformer([
        ("smri", build_smri_branch(n_smri), FS_COLS),
        ("fmri", build_fmri_branch(n_fmri), FM_COLS),
    ])

# --------------------------- HEADS ---------------------------
def head_linear(C=1.0):
    return make_calibrated_linear_svc(C=C, method="sigmoid", cv=CALIB_CV)

def head_nystrom_rbf(C=1.0, n_components=200, gamma=0.01):
    base = Pipeline([
        ("nyst", Nystroem(kernel="rbf", n_components=n_components, gamma=gamma, random_state=RANDOM_SEED)),
        ("svc",  LinearSVC(C=C, class_weight="balanced", random_state=RANDOM_SEED))
    ])
    sig = signature(CalibratedClassifierCV.__init__)
    if "estimator" in sig.parameters:
        return CalibratedClassifierCV(estimator=base, cv=CALIB_CV, method="sigmoid")
    else:
        return CalibratedClassifierCV(base_estimator=base, cv=CALIB_CV, method="sigmoid")

# --------------------------- DATA SLICERS ---------------------------
def subset_pair(fs, fm, y, a, b):
    mask = y.isin([a,b])
    idx  = y.index[mask]
    Xs   = fs.loc[idx, FS_COLS]
    Xf   = fm.loc[idx, FM_COLS]
    yy   = y.loc[idx].copy()
    yy = (yy == b).astype(int)
    return Xs, Xf, yy

def make_Xcat(Xs, Xf):
    return pd.concat([Xs, Xf], axis=1)

# --------------------------- GRID FACTORY ---------------------------
def grid_for_pair(pair):
    a,b = pair
    hard_pair = set(pair)==set(["AD","MCI"])
    if hard_pair:
        pipe = Pipeline([
            ("embed", make_embedder(20, 30)),
            ("clf",  head_nystrom_rbf())
        ])
        grid = {
            "embed__smri__pca__n_components": PCA_S_GRID,
            "embed__fmri__pca__n_components": PCA_F_GRID,
            "clf__estimator__nyst__n_components": NYST_NC,
            "clf__estimator__nyst__gamma": NYST_GAMMA,
            "clf__estimator__svc__C": C_GRID_RBF,
        }
    else:
        pipe = Pipeline([
            ("embed", make_embedder(20, 30)),
            ("clf",  head_linear())
        ])
        grid = {
            "embed__smri__pca__n_components": PCA_S_GRID,
            "embed__fmri__pca__n_components": PCA_F_GRID,
            "clf__estimator__C": C_GRID_LIN,
        }
    return pipe, grid

# --------------------------- FIT FINAL HEADS ---------------------------
best_models, best_configs = {}, {}

for (a,b) in PAIR_HEADS:
    print(f"\n[INFO] Fitting final head {a} vs {b} on ALL subjects ...")
    Xs, Xf, yy = subset_pair(fs, fm, y_all, a, b)
    X_cat = make_Xcat(Xs, Xf)

    pipe, grid = grid_for_pair((a,b))
    search = GridSearchCV(pipe, grid, cv=5, scoring="roc_auc", n_jobs=-1, refit=True)
    t0 = time.time()
    search.fit(X_cat, yy)
    print(f"[INFO] Best {a}vs{b}: {search.best_params_} | AUC={search.best_score_:.3f} | {time.time()-t0:.1f}s")
    best_models[(a,b)] = search.best_estimator_
    best_configs[f"{a}vs{b}"] = {"params": search.best_params_, "AUC": float(search.best_score_)}

with open(os.path.join(OUT_DIR, "best_final_heads.json"), "w") as f:
    json.dump(best_configs, f, indent=2)

# --------------------------- SHAP helpers ---------------------------
import shap

def _get_fitted_linear_from_calibrator(calibrated):
    return calibrated.calibrated_classifiers_[0].estimator

def _wrapped_estimator(clf_step):
    return _get_fitted_linear_from_calibrator(clf_step)

def get_embedded_matrix(best_estimator, X_cat):
    embed = best_estimator.named_steps["embed"]
    Z = embed.transform(X_cat)
    smri_pca = embed.named_transformers_["smri"].named_steps["pca"]
    fmri_pca = embed.named_transformers_["fmri"].named_steps["pca"]
    ks = smri_pca.n_components_
    kf = fmri_pca.n_components_
    cols = [f"sMRI_PC{i+1}" for i in range(ks)] + [f"fMRI_PC{i+1}" for i in range(kf)]
    Z_df = pd.DataFrame(Z, index=X_cat.index, columns=cols)
    return Z_df, smri_pca, fmri_pca

def shap_linear_on_embedding(best_estimator, X_cat, pair_name):
    Z_df, _, _ = get_embedded_matrix(best_estimator, X_cat)
    base = _wrapped_estimator(best_estimator.named_steps["clf"])
    explainer = shap.LinearExplainer(base, Z_df)
    shap_vals = explainer.shap_values(Z_df)
    pd.DataFrame(shap_vals, index=Z_df.index, columns=Z_df.columns).to_csv(
        os.path.join(OUT_DIR, f"shap_linear_{pair_name}_embedding.csv")
    )
    plt.figure(figsize=(8,5))
    shap.summary_plot(shap_vals, Z_df, show=False, max_display=20)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"shap_linear_{pair_name}_summary.png"), dpi=200)
    plt.close()

def shap_kernel_on_predict_proba(best_estimator, X_cat, pair_name):
    n_bg = min(BGNSZ, len(X_cat))
    X_bg = X_cat.sample(n_bg, random_state=RANDOM_SEED)
    f = lambda X: best_estimator.predict_proba(pd.DataFrame(X, columns=X_cat.columns))[:, 1]
    explainer = shap.KernelExplainer(f, X_bg)
    shap_vals = explainer.shap_values(X_cat, nsamples=NSHZ)
    pd.DataFrame(shap_vals, index=X_cat.index, columns=X_cat.columns).to_csv(
        os.path.join(OUT_DIR, f"shap_kernel_{pair_name}_rawspace.csv")
    )
    plt.figure(figsize=(8,5))
    shap.summary_plot(shap_vals, X_cat, show=False, max_display=20)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"shap_kernel_{pair_name}_summary.png"), dpi=200)
    plt.close()

# --------------------------- XAI RUN ---------------------------
print("\n[INFO] Running XAI (SHAP) per head ...")
for (a,b) in PAIR_HEADS:
    pair_name = f"{a}vs{b}"
    model = best_models[(a,b)]
    Xa, Xf, yy = subset_pair(fs, fm, y_all, a, b)
    X_cat = make_Xcat(Xa, Xf)
    if set((a,b))==set(["AD","MCI"]):
        print(f"[XAI] Kernel SHAP for {pair_name} ...")
        shap_kernel_on_predict_proba(model, X_cat, pair_name)
    else:
        print(f"[XAI] Linear SHAP for {pair_name} ...")
        shap_linear_on_embedding(model, X_cat, pair_name)

print(f"\n[DONE] Artifacts saved to: {OUT_DIR}")


## Visualizing the results

In [ ]:
# Set this path to your timestamped run folder (the one that contains /metrics).
RUN_DIR = r"H:\Master's Thesis\Results\Multimodal_Hybrid_OvO_PlusPlus\20251011_151442"

import os, glob, numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score, f1_score, accuracy_score
from sklearn.calibration import calibration_curve

# ---------- Styling (IEEE-friendly) ----------
mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ---------- I/O ----------
METRICS_DIR = os.path.join(RUN_DIR, "metrics")
FIG_DIR = os.path.join(RUN_DIR, "figures"); os.makedirs(FIG_DIR, exist_ok=True)

def _load_preds(metrics_dir):
    paths = sorted(glob.glob(os.path.join(metrics_dir, "fold_*_predictions.csv")))
    if not paths: raise FileNotFoundError(f"No fold_*_predictions.csv in {metrics_dir}")
    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        need = {"true","pred","proba_AD","proba_CN","proba_MCI"}
        if not need.issubset(df.columns): raise ValueError(f"Missing columns in {p} → need {need}")
        # fold index from file name
        try:
            fold = int(os.path.basename(p).split("_")[1])
        except Exception:
            fold = len(dfs) + 1
        df["fold"] = fold
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

agg = pd.read_csv(os.path.join(METRICS_DIR, "aggregate_metrics.csv"))
agg_pw = pd.read_csv(os.path.join(METRICS_DIR, "aggregate_pairwise_metrics.csv"))
folds = pd.read_csv(os.path.join(METRICS_DIR, "fold_metrics.csv"))
preds = _load_preds(METRICS_DIR)

# ---------- Prepare arrays ----------
CLASSES = ["AD","CN","MCI"]  # expected order in your files
y_true = preds["true"].astype(str).values
y_pred = preds["pred"].astype(str).values
probs = preds[["proba_AD","proba_CN","proba_MCI"]].values

# ---------- Utility metrics ----------
def one_vs_rest_curves(y_true, probs, cls):
    yb = (y_true == cls).astype(int)
    fpr, tpr, _ = roc_curve(yb, probs)
    roc_auc = auc(fpr, tpr)
    prec, rec, _ = precision_recall_curve(yb, probs)
    ap = average_precision_score(yb, probs)
    return fpr, tpr, roc_auc, rec, prec, ap

def macro_scores(y_true, probs, classes=CLASSES):
    y_hat = np.array(classes)[np.argmax(probs, axis=1)]
    acc = accuracy_score(y_true, y_hat)
    f1m = f1_score(y_true, y_hat, average="macro")
    roc_list, pr_list = [], []
    for i, c in enumerate(classes):
        yb = (y_true == c).astype(int)
        fpr, tpr, _ = roc_curve(yb, probs[:, i])
        roc_list.append(auc(fpr, tpr))
        pr_list.append(average_precision_score(yb, probs[:, i]))
    return acc, f1m, float(np.mean(roc_list)), float(np.mean(pr_list))

acc_r, f1m_r, rocM_r, prM_r = macro_scores(y_true, probs)
print(f"[Recomputed] Acc={acc_r:.3f} | Macro-F1={f1m_r:.3f} | ROC-AUC_macro={rocM_r:.3f} | PR-AUC_macro={prM_r:.3f}")

# ---------- 1) Confusion Matrix (normalized) ----------
cm = confusion_matrix(y_true, y_pred, labels=CLASSES).astype(float)
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(3.6, 3.2))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(cmn.shape[0]):
    for j in range(cmn.shape[1]):
        ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", color="black")
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix (normalized)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()
fig.savefig(os.path.join(FIG_DIR, "fig_confusion_matrix.png"))
fig.savefig(os.path.join(FIG_DIR, "fig_confusion_matrix.pdf"))
plt.close(fig)

# ---------- 2) ROC (One-vs-Rest) ----------
fig, ax = plt.subplots(figsize=(3.6, 3.0))
for i, c in enumerate(CLASSES):
    fpr, tpr, roc_auc, *_ = one_vs_rest_curves(y_true, probs[:, i], c)
    ax.plot(fpr, tpr, label=f"{c} (AUC={roc_auc:.3f})")
ax.plot([0,1],[0,1],"--",color="gray",linewidth=1)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC (One-vs-Rest)"); ax.legend(loc="lower right", frameon=False)
fig.tight_layout(); plt.show()
fig.savefig(os.path.join(FIG_DIR, "fig_roc_ovr.png"))
fig.savefig(os.path.join(FIG_DIR, "fig_roc_ovr.pdf"))
plt.close(fig)

# ---------- 3) Precision–Recall (OvR) ----------
fig, ax = plt.subplots(figsize=(3.6, 3.0))
aps = []
for i, c in enumerate(CLASSES):
    _, _, _, rec, prec, ap = one_vs_rest_curves(y_true, probs[:, i], c)
    ax.plot(rec, prec, label=f"{c} (AP={ap:.3f})")
    aps.append(ap)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"Precision–Recall (OvR) | Macro AP={np.mean(aps):.3f}")
ax.legend(loc="lower left", frameon=False)
fig.tight_layout(); plt.show()
fig.savefig(os.path.join(FIG_DIR, "fig_pr_ovr.png"))
fig.savefig(os.path.join(FIG_DIR, "fig_pr_ovr.pdf"))
plt.close(fig)

# ---------- 4) Calibration (Reliability) ----------
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6), sharex=True, sharey=True)
for i, c in enumerate(CLASSES):
    yb = (y_true == c).astype(int)
    frac_pos, mean_pred = calibration_curve(yb, probs[:, i], n_bins=10, strategy="uniform")
    axes[i].plot(mean_pred, frac_pos, marker="o", linewidth=1.5, label="Empirical")
    axes[i].plot([0,1],[0,1],"--",color="gray",linewidth=1)
    axes[i].set_title(f"Calibration: {c}")
    axes[i].set_xlabel("Predicted probability")
    if i == 0: axes[i].set_ylabel("Empirical frequency")
fig.suptitle("Reliability Curves (Platt-calibrated)", y=0.98, fontsize=10)
fig.tight_layout(); plt.show()
fig.savefig(os.path.join(FIG_DIR, "fig_calibration_curves.png"))
fig.savefig(os.path.join(FIG_DIR, "fig_calibration_curves.pdf"))
plt.close(fig)

# ---------- 5) Per-fold bars (Acc, Macro-F1, OvO-AUC, PR-AUC, Brier) ----------
metrics = ["accuracy","f1_macro","roc_auc_ovo","pr_auc_macro","brier_mean"]
titles  = ["Accuracy","Macro F1","ROC AUC (OvO)","PR AUC (macro)","Brier (↓)"]
fig, axes = plt.subplots(1, len(metrics), figsize=(7.4, 2.3))
for ax, m, t in zip(axes, metrics, titles):
    ax.bar(folds["fold"].astype(str), folds[m], width=0.65)
    ax.set_title(t); ax.set_xlabel("Fold")
    if m != "brier_mean": ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
    else: ax.set_ylabel("Brier")
    for x, y in zip(folds["fold"].astype(str), folds[m]): ax.text(x, y+(0.01 if m!="brier_mean" else 0.0), f"{y:.2f}", ha="center", va="bottom", fontsize=8)
fig.tight_layout(); plt.show()
fig.savefig(os.path.join(FIG_DIR, "fig_per_fold_bars.png"))
fig.savefig(os.path.join(FIG_DIR, "fig_per_fold_bars.pdf"))
plt.close(fig)

# ---------- 6) Probability histograms (diagnostic) ----------
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.2), sharey=True)
for i, c in enumerate(CLASSES):
    axes[i].hist(probs[:, i], bins=20)
    axes[i].set_title(f"P({c}) distribution")
    axes[i].set_xlabel("Probability")
    if i == 0: axes[i].set_ylabel("Count")
fig.tight_layout(); plt.show()
fig.savefig(os.path.join(FIG_DIR, "fig_probability_histograms.png"))
fig.savefig(os.path.join(FIG_DIR, "fig_probability_histograms.pdf"))
plt.close(fig)

# ---------- 7) Optional: Unimodal vs Multimodal (if sibling runs found) ----------
siblings = glob.glob(os.path.join(os.path.dirname(RUN_DIR), "*"))
labels, vals = [], []
for s in siblings:
    am = os.path.join(s, "metrics", "aggregate_metrics.csv")
    if os.path.exists(am):
        df = pd.read_csv(am).set_index("metric")
        if {"accuracy","f1_macro","roc_auc_ovo"}.issubset(df.index):
            labels.append(os.path.basename(s))
            vals.append([df.loc["accuracy","mean"], df.loc["f1_macro","mean"], df.loc["roc_auc_ovo","mean"]])
if vals:
    vals = np.array(vals)
    fig, ax = plt.subplots(figsize=(4.8, 2.8))
    idx = np.arange(len(labels)); w = 0.25
    ax.bar(idx - w, vals[:,0], width=w, label="Acc")
    ax.bar(idx,       vals[:,1], width=w, label="F1")
    ax.bar(idx + w,   vals[:,2], width=w, label="ROC AUC (OvO)")
    ax.set_xticks(idx); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Score"); ax.set_title("Unimodal vs. Multimodal (if found)")
    ax.legend(frameon=False, ncols=3)
    fig.tight_layout(); plt.show()
    fig.savefig(os.path.join(FIG_DIR, "fig_unimodal_vs_multimodal.png"))
    fig.savefig(os.path.join(FIG_DIR, "fig_unimodal_vs_multimodal.pdf"))
    plt.close(fig)

# ---------- 8) Save a short text summary (appendix helper) ----------
with open(os.path.join(FIG_DIR, "summary.txt"), "w", encoding="utf-8") as f:
    f.write("=== Aggregate metrics (aggregate_metrics.csv) ===\n")
    f.write(agg.to_string(index=False) + "\n\n")
    f.write("=== Pairwise head metrics (aggregate_pairwise_metrics.csv) ===\n")
    f.write(agg_pw.to_string(index=False) + "\n\n")
    f.write(f"[Recomputed] Acc={acc_r:.3f} | Macro-F1={f1m_r:.3f} | ROC-AUC_macro={rocM_r:.3f} | PR-AUC_macro={prM_r:.3f}\n")

print(f"[DONE] Figures saved to: {FIG_DIR}")


In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============ CONFIG ============
CSV_PATH = r"C:\Users\Alireza217\Desktop\subjects_used.csv"   # <-- change to your CSV path
OUT_DIR  = r"C:\Users\Alireza217\Desktop\demographics"                    # e.g., r"H:\Master’s Thesis\figures"
DPI      = 300                             # export resolution
# =================================

os.makedirs(OUT_DIR, exist_ok=True)

# Matplotlib style: simple, journal-friendly
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.dpi": DPI
})

# ---------- Load & normalize ----------
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

def pick_col(possible_names, cols):
    for name in possible_names:
        if name in cols:
            return name
    for name in possible_names:
        for c in cols:
            if name in c:
                return c
    return None

age_col = pick_col(["age", "age_years"], df.columns)
sex_col = pick_col(["sex", "gender", "biological_sex"], df.columns)
dx_col  = pick_col(["diagnosis", "dx", "group", "label", "class"], df.columns)
site_col= pick_col(["site", "scanner", "center", "centre"], df.columns)

# ---------- Light cleaning ----------
def clean_sex(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower()
    if s in {"m","male","1"}: return "Male"
    if s in {"f","female","0"}: return "Female"
    return str(x).strip()

def clean_dx(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().upper()
    mapping = {
        "CN":"CN","CONTROL":"CN","NORMAL":"CN","HC":"CN",
        "MCI":"MCI","EMCI":"MCI","LMCI":"MCI",
        "SMC":"SMC",
        "AD":"AD","DEMENTIA":"AD"
    }
    return mapping.get(s, s)

if sex_col:
    df[sex_col] = df[sex_col].apply(clean_sex)
if dx_col:
    df[dx_col]  = df[dx_col].apply(clean_dx)

# Nice diagnosis order for ADNI-like cohorts
dx_order = None
if dx_col:
    base = ["CN","SMC","MCI","AD"]
    present = [d for d in base if d in set(df[dx_col].dropna())]
    others  = sorted([d for d in df[dx_col].dropna().unique() if d not in present])
    dx_order = present + others if present or others else None

# Utility: save both PNG + SVG
def savefig_both(path_base):
    png = os.path.join(OUT_DIR, f"{path_base}.png")
    svg = os.path.join(OUT_DIR, f"{path_base}.svg")
    plt.tight_layout()
    plt.savefig(png, dpi=DPI, bbox_inches="tight")
    plt.savefig(svg, dpi=DPI, bbox_inches="tight")
    print(f"Saved: {png}\n       {svg}")

# ---------- Figure A: Age histogram ----------
if age_col:
    ages = pd.to_numeric(df[age_col], errors="coerce").dropna()
    fig = plt.figure(figsize=(4.0, 3.0))
    n_bins = min(20, max(8, int(np.sqrt(len(ages))) if len(ages)>0 else 10))
    plt.hist(ages, bins=n_bins)
    plt.xlabel("Age (years)")
    plt.ylabel("Count")
    plt.title("Age distribution")
    # optional reference lines
    if len(ages) > 1:
        m, sd = ages.mean(), ages.std(ddof=1)
        plt.axvline(m, linestyle="--")
        plt.text(m, plt.ylim()[1]*0.9, f"mean={m:.1f}", ha="center", va="top")
    savefig_both("age_distribution")
    plt.close(fig)

# ---------- Figure B: Sex distribution ----------
if sex_col:
    counts = df[sex_col].value_counts(dropna=False)
    fig = plt.figure(figsize=(3.6, 3.0))
    counts.plot(kind="bar")
    plt.xlabel("Sex")
    plt.ylabel("Count")
    plt.title("Sex distribution")
    savefig_both("sex_distribution")
    plt.close(fig)

# ---------- Figure C: Diagnosis distribution ----------
if dx_col:
    vc = df[dx_col].value_counts(dropna=False)
    if dx_order:
        vc = vc.reindex(dx_order).fillna(0).astype(int)
    fig = plt.figure(figsize=(4.0, 3.0))
    vc.plot(kind="bar")
    plt.xlabel("Diagnosis")
    plt.ylabel("Count")
    plt.title("Diagnosis distribution")
    savefig_both("diagnosis_distribution")
    plt.close(fig)

# ---------- Figure D: Site distribution ----------
if site_col:
    vc_site = df[site_col].value_counts(dropna=False)
    # collapse long tails into "Other" if too many categories
    if len(vc_site) > 20:
        top = vc_site.head(19)
        other = pd.Series({"Other": vc_site.iloc[19:].sum()})
        vc_site = pd.concat([top, other])
    fig = plt.figure(figsize=(5.0, 3.0))
    vc_site.plot(kind="bar")
    plt.xlabel("Site")
    plt.ylabel("Count")
    plt.title("Site distribution")
    savefig_both("site_distribution")
    plt.close(fig)

# ---------- (Optional) Figure E: Age by diagnosis (boxplot) ----------
if age_col and dx_col and df[dx_col].notna().any():
    tmp = df[[dx_col, age_col]].copy()
    tmp[age_col] = pd.to_numeric(tmp[age_col], errors="coerce")
    tmp = tmp.dropna()
    if not tmp.empty:
        groups = dx_order if dx_order else sorted(tmp[dx_col].dropna().unique())
        data = [tmp.loc[tmp[dx_col]==g, age_col].values for g in groups]
        fig = plt.figure(figsize=(4.2, 3.0))
        plt.boxplot(data, labels=groups, showfliers=False)
        plt.xlabel("Diagnosis")
        plt.ylabel("Age (years)")
        plt.title("Age by diagnosis")
        savefig_both("age_by_diagnosis_boxplot")
        plt.close(fig)

# ---------- Console summary ----------
print("\n=== Detected columns ===")
print(f"Age:       {age_col}")
print(f"Sex:       {sex_col}")
print(f"Diagnosis: {dx_col}")
print(f"Site:      {site_col}")
print(f"\nFigures saved to: {OUT_DIR}")
